In [0]:
# ============================================================
# Configuration
# ============================================================

CATALOG = "worldbank_ai"

SOURCE_TABLE = (
    f"{CATALOG}.gold.macroeconomic_indicators"
)

TARGET_TABLE = (
    f"{CATALOG}.gold.country_summary"
)

print(f"Source: {SOURCE_TABLE}")
print(f"Target: {TARGET_TABLE}")

In [0]:
# ============================================================
# Imports
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# ============================================================
# Load Gold macroeconomic observations
# ============================================================

macro_df = spark.table(
    SOURCE_TABLE
)

macro_count = macro_df.count()

print(f"Macroeconomic records: {macro_count:,}")

In [0]:
# ============================================================
# Define headline indicators
# ============================================================
#
# country_summary is intentionally a compact snapshot.
#
# Detailed historical queries should continue to use:
#
#   gold.macroeconomic_indicators
#
# This table is for quick entity-level context.
# ============================================================

HEADLINE_INDICATORS = {

    "NY.GDP.MKTP.KD.ZG":
        "gdp_growth",

    "NY.GDP.PCAP.KD.ZG":
        "gdp_per_capita_growth",

    "NY.GDP.MKTP.CD":
        "gdp_current_usd",

    "NY.GDP.PCAP.CD":
        "gdp_per_capita_current_usd",

    "FP.CPI.TOTL.ZG":
        "inflation",

    "NE.TRD.GNFS.ZS":
        "trade_pct_gdp",

    "BX.KLT.DINV.WD.GD.ZS":
        "fdi_net_inflows_pct_gdp",

    "BN.CAB.XOKA.GD.ZS":
        "current_account_pct_gdp"
}

headline_ids = list(
    HEADLINE_INDICATORS.keys()
)

print(
    f"Configured headline indicators: "
    f"{len(headline_ids)}"
)

In [0]:
# ============================================================
# Find latest available value per entity + indicator
# ============================================================
#
# IMPORTANT:
#
# Indicators have different publication/reporting lags.
#
# Therefore:
#
# GDP growth might have a 2025 value,
# while another indicator might only have a 2023 value.
#
# We independently find the latest NON-NULL value for every
# entity + indicator pair.
# ============================================================

headline_df = (
    macro_df

    .filter(
        F.col("indicator_id").isin(
            headline_ids
        )
    )

    .filter(
        F.col("value").isNotNull()
    )
)


latest_window = (
    Window

    .partitionBy(
        "entity_id",
        "indicator_id"
    )

    .orderBy(
        F.col("year").desc()
    )
)


latest_df = (
    headline_df

    .withColumn(
        "latest_rank",

        F.row_number().over(
            latest_window
        )
    )

    .filter(
        F.col("latest_rank") == 1
    )

    .drop(
        "latest_rank"
    )
)

print(
    f"Latest entity-indicator observations: "
    f"{latest_df.count():,}"
)

In [0]:
# ============================================================
# Validate latest observation grain
# ============================================================

latest_duplicate_count = (
    latest_df

    .groupBy(
        "entity_id",
        "indicator_id"
    )

    .count()

    .filter(
        F.col("count") > 1
    )

    .count()
)

print(
    f"Duplicate latest entity-indicator records: "
    f"{latest_duplicate_count}"
)

if latest_duplicate_count != 0:
    raise RuntimeError(
        "Duplicate latest entity-indicator records detected."
    )

print("Latest observation grain validated.")

In [0]:
# ============================================================
# Create one-row-per-entity base
# ============================================================

entity_base_df = (
    macro_df

    .select(
        "entity_id",
        "iso2_code",
        "entity_name",
        "entity_type",

        "region_id",
        "region_name",

        "income_level_id",
        "income_level_name",

        "lending_type_id",
        "lending_type_name"
    )

    .distinct()
)

entity_count = entity_base_df.count()

print(
    f"Observed entities: {entity_count:,}"
)

if entity_count != 265:
    raise RuntimeError(
        f"Expected 265 entities, found {entity_count}."
    )

In [0]:
# ============================================================
# Pivot latest indicator VALUES
# ============================================================
#
# Before:
#
# IND | GDP growth | 2025 | value
# IND | Inflation  | 2024 | value
#
# After:
#
# IND | gdp_growth_value | inflation_value | ...
# ============================================================

value_pivot_df = (
    latest_df

    .groupBy(
        "entity_id"
    )

    .pivot(
        "indicator_id",
        headline_ids
    )

    .agg(
        F.first("value")
    )
)

In [0]:
# ============================================================
# Pivot corresponding observation YEARS
# ============================================================
#
# We keep the year for every metric so an agent cannot present
# a 2022 value as though it were a 2025 value.
# ============================================================

year_pivot_df = (
    latest_df

    .groupBy(
        "entity_id"
    )

    .pivot(
        "indicator_id",
        headline_ids
    )

    .agg(
        F.first("year")
    )
)

In [0]:
# ============================================================
# Rename generated pivot columns
# ============================================================

for indicator_id, prefix in HEADLINE_INDICATORS.items():

    # Example:
    #
    # NY.GDP.MKTP.KD.ZG
    #      ->
    # gdp_growth_value

    if indicator_id in value_pivot_df.columns:

        value_pivot_df = (
            value_pivot_df
            .withColumnRenamed(
                indicator_id,
                f"{prefix}_value"
            )
        )

    # Corresponding observation year:
    #
    # gdp_growth_year

    if indicator_id in year_pivot_df.columns:

        year_pivot_df = (
            year_pivot_df
            .withColumnRenamed(
                indicator_id,
                f"{prefix}_year"
            )
        )

In [0]:
# ============================================================
# Build final country/entity summary
# ============================================================

country_summary_df = (
    entity_base_df

    .join(
        value_pivot_df,
        on="entity_id",
        how="left"
    )

    .join(
        year_pivot_df,
        on="entity_id",
        how="left"
    )

    .withColumn(
        "gold_processed_at",
        F.current_timestamp()
    )
)

In [0]:
# ============================================================
# Add headline-indicator availability information
# ============================================================

value_columns = [
    f"{prefix}_value"
    for prefix in HEADLINE_INDICATORS.values()
]


# Count how many headline metrics are available for each entity.
availability_expression = sum(

    (
        F.when(
            F.col(column_name).isNotNull(),
            1
        ).otherwise(0)
    )

    for column_name in value_columns
)


country_summary_df = (
    country_summary_df

    .withColumn(
        "headline_indicators_available",
        availability_expression
    )

    .withColumn(
        "headline_indicators_configured",
        F.lit(
            len(HEADLINE_INDICATORS)
        )
    )

    .withColumn(
        "headline_coverage_pct",

        F.round(
            F.col("headline_indicators_available")
            / F.col("headline_indicators_configured")
            * 100,
            2
        )
    )
)

In [0]:
# ============================================================
# Validate one-row-per-entity grain
# ============================================================

summary_count = (
    country_summary_df.count()
)

summary_distinct_entities = (
    country_summary_df
    .select("entity_id")
    .distinct()
    .count()
)


print(f"Summary rows:      {summary_count:,}")
print(f"Distinct entities: {summary_distinct_entities:,}")


if summary_count != summary_distinct_entities:
    raise RuntimeError(
        "Country summary contains duplicate entities."
    )

if summary_count != 265:
    raise RuntimeError(
        f"Expected 265 entity summaries, found {summary_count}."
    )

print(
    "Country summary grain validation passed."
)

In [0]:
# ============================================================
# Inspect headline indicator coverage
# ============================================================

display(
    country_summary_df

    .groupBy(
        "headline_indicators_available"
    )

    .count()

    .orderBy(
        "headline_indicators_available"
    )
)

In [0]:
# ============================================================
# Sanity check: India economic snapshot
# ============================================================

display(
    country_summary_df

    .filter(
        F.col("entity_id") == "IND"
    )
)

In [0]:
# ============================================================
# Persist Gold country/entity summary
# ============================================================

(
    country_summary_df

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        TARGET_TABLE
    )
)

print(
    f"Saved Gold table: {TARGET_TABLE}"
)

In [0]:
# ============================================================
# Validate persisted country summary
# ============================================================

saved_summary_df = spark.table(
    TARGET_TABLE
)

saved_count = saved_summary_df.count()

saved_distinct_entities = (
    saved_summary_df
    .select("entity_id")
    .distinct()
    .count()
)


print(f"Expected rows:     {summary_count:,}")
print(f"Saved rows:        {saved_count:,}")
print(f"Distinct entities: {saved_distinct_entities:,}")


if saved_count != summary_count:
    raise RuntimeError(
        "Country summary write row-count validation failed."
    )

if saved_distinct_entities != saved_count:
    raise RuntimeError(
        "Country summary entity uniqueness validation failed."
    )

print(
    "Country summary write validation passed."
)